In [1]:
%%time

import warnings
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor
from sklearn.svm import SVR
from sklearn.compose import make_column_transformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, make_scorer, mean_absolute_error
from sklearn.pipeline import make_pipeline,make_union
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.preprocessing import PolynomialFeatures, PowerTransformer, FunctionTransformer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import cross_val_score, KFold, ShuffleSplit, learning_curve, GridSearchCV
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import DBSCAN
import category_encoders as ce

from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

warnings.filterwarnings('ignore')
pd.set_option("display.max_column",100)


Wall time: 1min 13s


In [175]:
train_df = pd.read_csv('data/Train.csv')
train_df.drop(["ID"], axis=1, inplace=True)

In [176]:
train_df.head()

,District,Block,CultLand,CropCultLand,LandPreparationMethod,CropTillageDate,CropTillageDepth,CropEstMethod,RcNursEstDate,SeedingSowingTransplanting,SeedlingsPerPit,NursDetFactor,TransDetFactor,TransplantingIrrigationHours,TransplantingIrrigationSource,TransplantingIrrigationPowerSource,TransIrriCost,StandingWater,OrgFertilizers,Ganaura,CropOrgFYM,PCropSolidOrgFertAppMethod,NoFertilizerAppln,CropbasalFerts,BasalDAP,BasalUrea,MineralFertAppMethod,FirstTopDressFert,1tdUrea,1appDaysUrea,2tdUrea,2appDaysUrea,MineralFertAppMethod.1,Harv_method,Harv_date,Harv_hand_rent,Threshing_date,Threshing_method,Residue_length,Residue_perc,Stubble_use,Acre,Yield
0,Nalanda,Noorsarai,45,40,TractorPlough FourWheelTracRotavator,2022-07-20,5,Manual_PuddledRandom,2022-06-27,2022-07-21,2.0,CalendarDate IrrigWaterAvailability SeedAvaila...,CalendarDate SeedlingAge RainArrival IrrigWate...,5.0,Boring,Electric,200.0,2.0,NaN,NaN,NaN,NaN,2,Urea,NaN,20.0,Broadcasting,Urea,15.0,18.0,NaN,NaN,Broadcasting,machine,2022-11-16,NaN,2022-11-16,machine,30,40,plowed_in_soil,0.312500,600
1,Nalanda,Rajgir,26,26,WetTillagePuddling TractorPlough FourWheelTrac...,2022-07-18,5,Manual_PuddledRandom,2022-06-20,2022-07-20,2.0,CalendarDate PreMonsoonShowers IrrigWaterAvail...,CalendarDate SeedlingAge RainArrival IrrigWate...,5.0,Boring,Electric,125.0,3.0,NaN,NaN,NaN,NaN,2,DAP Urea,15.0,10.0,Broadcasting,Urea,20.0,39.0,NaN,NaN,Broadcasting,hand,2022-11-25,3.0,2022-12-24,machine,24,10,plowed_in_soil,0.312500,600
2,Gaya,Gurua,10,10,TractorPlough FourWheelTracRotavator,2022-06-30,6,Manual_PuddledRandom,2022-06-20,2022-08-13,2.0,PreMonsoonShowers IrrigWaterAvailability Labou...,SeedlingAge IrrigWaterAvailability LaborAvaila...,4.0,Boring,Electric,80.0,2.0,Ganaura FYM,1.0,1.0,SoilApplied,2,DAP,4.0,NaN,SoilApplied,Urea,5.0,65.0,NaN,NaN,RootApplication,hand,2022-12-12,480.0,2023-01-11,machine,30,10,plowed_in_soil,0.148148,225
3,Gaya,Gurua,15,15,TractorPlough FourWheelTracRotavator,2022-06-16,6,Manual_PuddledRandom,2022-06-17,2022-07-17,2.0,CalendarDate PreMonsoonShowers IrrigWaterAvail...,CalendarDate SeedlingAge RainArrival IrrigWate...,NaN,NaN,NaN,NaN,NaN,Ganaura,1.0,NaN,SoilApplied,2,DAP Urea,6.0,3.0,Broadcasting,Urea,5.0,5.0,NaN,NaN,Broadcasting,hand,2022-12-02,240.0,2022-12-29,hand,26,10,plowed_in_soil,0.222222,468
4,Nalanda,Noorsarai,60,60,TractorPlough WetTillagePuddling,2022-07-19,4,Manual_PuddledRandom,2022-06-21,2022-07-20,2.0,CalendarDate IrrigWaterAvailability SeedAvaila...,SeedlingAge RainArrival IrrigWaterAvailability...,9.0,Boring,Electric,300.0,2.0,NaN,NaN,NaN,NaN,2,Urea DAP,15.0,30.0,Broadcasting,Urea,30.0,26.0,NaN,NaN,Broadcasting,machine,2022-11-30,NaN,2022-12-02,machine,24,40,plowed_in_soil,0.468750,550


In [177]:
df = train_df.copy()

In [178]:
anom = df.Yield.apply(lambda x: x>np.quantile(df.Yield, 0.88) or x<np.quantile(df.Yield, 0.05)).map({True:"Anormal", False:"Normal"})

In [179]:
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)


RMSE=make_scorer(rmse)

In [180]:
def create_date_features(data, feature):
    return pd.DatetimeIndex(data[feature]).month.map(str)


def create_duration_features(data, features):
    duration_df = pd.DataFrame()
    data[features]=data[features].apply(pd.to_datetime)
    patterns=[i.split("/") for i in (set(["/".join(sorted(list(set([i,j])))) for i in features for j in features if len(set([i,j]))>1]))]
    for i in patterns:
        duration_df[f"{i[0]}-{i[1]}"]=(data[i[1]]-data[i[0]]).dt.days
    return duration_df


def fill_implicit_missing(df):
    df.Ganaura = np.where((df.OrgFertilizers.notna() & df.OrgFertilizers.apply(lambda x: ("ganaura" not in str(x).lower().split()))), 0, df.Ganaura.values)
    df.CropOrgFYM = np.where((df.OrgFertilizers.notna() & df.OrgFertilizers.apply(lambda x: ("fym" not in str(x).lower().split()))), 0, df.CropOrgFYM.values)
    df.BasalUrea = np.where((df.CropbasalFerts.notna() & df.CropbasalFerts.apply(lambda x: ("urea" not in str(x).lower().split()))), 0, df.BasalUrea.values)
    df.BasalDAP = np.where((df.CropbasalFerts.notna() & df.CropbasalFerts.apply(lambda x: ("dap" not in str(x).lower().split()))), 0, df.BasalDAP.values)
    df["1tdUrea"] = np.where(df.NoFertilizerAppln < 2, -999, df["1tdUrea"].values)
    df["1appDaysUrea"] = np.where(df.NoFertilizerAppln < 2, -999, df["1appDaysUrea"].values)
    df["2tdUrea"] = np.where(df.NoFertilizerAppln < 3, -999, df["2tdUrea"].values)
    df["2appDaysUrea"] = np.where(df.NoFertilizerAppln < 3 , -999, df["2appDaysUrea"].values)

    return df


In [181]:
def create_new_features(df):
    features=["CropTillageDate","RcNursEstDate","SeedingSowingTransplanting","Harv_date","Threshing_date"]

    df["CropTillageMonth"]=create_date_features(df, "CropTillageDate")
    df["RcNursEstMonth"]=create_date_features(df, "RcNursEstDate")
    df["SeedingSowingTransplantingMonth"]=create_date_features(df,"SeedingSowingTransplanting")
    df["Harv_Month"]=create_date_features(df, "Harv_date")
    # df["Threshing_Month"]=create_date_features(df, "Threshing_date")

    duration_df=create_duration_features(df.copy(), features)

    df = pd.concat([df, duration_df.drop('Harv_date-RcNursEstDate', axis=1)], axis=1)

    columns = ["LandPreparationMethod", "FirstTopDressFert", "CropbasalFerts", "TransDetFactor", "NursDetFactor"]

    for column in columns:
        n_column = df[column].map(str).apply(lambda x: x.split())
        df[f"num_{column}"]=n_column.apply(lambda x: len(x))
        # alpha = df[f"num_{column}"].max()
        # n_column = n_column.apply(lambda x: x+[np.nan]*(alpha-len(x)))
        # df_applied = n_column.apply(lambda row: pd.Series({f"{column}_{i+1}":row[i] for i in range(alpha)}))
        # df = pd.concat([df.drop(column, axis=1), df_applied],axis=1)

    # df.dropna(subset=["CropOrgFYM"], axis=0)

    # df = df[df.columns[df.isna().sum()/df.shape[0] < 0.3 ]]

    # df = df.select_dtypes(np.number)

    return df

In [182]:
def imputation(df):

    # df_tr = MissingIndicator().fit_transform(df)
    # df = pd.concat([df,df_tr], axis=1)

    df = fill_implicit_missing(df)

    dt=(df[["1tdUrea","1appDaysUrea","2tdUrea","2appDaysUrea"]] == -999).astype(int)
    dt.rename(columns={"1tdUrea":"1tdUrea_M","1appDaysUrea":"1appDaysUrea_2","2tdUrea":"2tdUrea_M","2appDaysUrea":"2appDaysUrea_M"}, inplace=True)

    df = pd.concat([df,dt],axis=1)
    
    ### HANDLING MISSING DATA ###

    cat_features = df.select_dtypes(np.object_).columns.to_list()
    num_features = df.select_dtypes(np.number).columns.to_list()
    #num_features.remove("Yield")

    df[cat_features]=df[cat_features].fillna(df.mode().iloc[0])
    df = df.fillna(0)

    return df

In [183]:
def encodage(df):
    
    
    return df

In [184]:
def pre_processing(df):
    
    df=imputation(df)
    df=create_new_features(df)
    df=encodage(df)

    return df

    # if "Yield" in df.columns.values:
    #     X = df.drop("Yield", axis=1)
    #     y = df.Yield
    #     return X,y
    # else:
    #     X = df
    #     return X

In [185]:

# train,test=train_test_split(df, test_size=0.2)
# X_train,y_train=pre_processing(train)
# X_test,y_test=pre_processing(test)

# X,y=pre_processing(df)

df = pre_processing(df)

In [186]:
catboost_encoder_df = ce.CatBoostEncoder(cols=df.select_dtypes(np.object_).columns.to_list())

df_encoded = catboost_encoder_df.fit_transform(df.drop(["Yield"], axis=1),df.Yield)

## Anomalie detection

### LoF

In [187]:
lof = LocalOutlierFactor(n_neighbors=55, contamination=0.2, novelty=True)  # Paramètres à ajuster
lof.fit(df_encoded)
predicted_labels_1 = lof.predict(df_encoded)

detected_anamalies = anom[predicted_labels_1==-1]

detected_anamalies.value_counts()

Normal     547
Anormal    198
Name: Yield, dtype: int64

### DBSCAN

In [188]:
# Entraînement du modèle DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=5)  # Paramètres à ajuster
predicted_labels_2 = dbscan.fit_predict(df_encoded)

# Extraction des anomalies prédites (points étiquetés comme -1 sont considérés comme des anomalies)
detected_anomalies = anom[predicted_labels_2 == -1]

detected_anamalies.value_counts()

Normal     547
Anormal    198
Name: Yield, dtype: int64

In [189]:
#df["Type"]=pd.Series(predicted_labels_1).map({-1:"Anormal", 1:"Normal"})

In [190]:
X_n = df[predicted_labels_1==1].drop(["Yield"], axis=1)
y_n = df[predicted_labels_1==1].Yield

In [191]:
X_a = df[predicted_labels_1==-1].drop(["Yield"], axis=1)
y_a = df[predicted_labels_1==-1].Yield

## Modelisation

In [192]:
def rmse_lgb(y_pred, y_true):
    
    rate=mean_squared_error(y_true, np.expm1(y_pred), squared=False)
    
    return "rmse_lgb", rate, False

In [193]:
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)


RMSE=make_scorer(rmse)

In [194]:
RANDOM_SEED=0
NUM_THREADS=0
MODEL_PARAMS_LGB = {
    "boosting_type": 'gbdt',
    "objective": "regression",  # <= set objective
    "metric":None,
    "seed": RANDOM_SEED,
    "verbose": -1,
}   
FIT_PARAMS_LGB = {"early_stopping_rounds": 100, "verbose_eval":100}

catboost_encoder = ce.CatBoostEncoder(cols=X_n.select_dtypes(np.object_).columns.to_list())
X_n_encode = catboost_encoder_df.fit_transform(X_n,y_n)

X_a_encode = catboost_encoder_df.transform(X_a,y_a)

cv=KFold(n_splits=10,shuffle=True,random_state=RANDOM_SEED)

In [195]:
scores=[]
for i,j in enumerate(cv.split(X_n_encode,y_n)):
    model=LGBMRegressor(boosting_type='gbdt', objective='regression',seed=RANDOM_SEED)
    train_X=X_n_encode.iloc[j[0]] 
    train_y=np.log1p(y_n.iloc[j[0]])
    test_X=X_n_encode.iloc[j[1]]
    test_y=np.log1p(y_n.iloc[j[1]])
    model.fit(train_X,train_y,
        eval_names=['train', 'valid'],
        eval_set=[(train_X,train_y),(test_X, test_y)], 
        eval_metric=rmse_lgb,
        verbose = False)
    
    y_pred=model.predict(X_n_encode.iloc[j[1]])
    score=rmse(np.expm1(y_pred),y_n.iloc[j[1]])
    scores.append(score)
    print(f"Fold {i}\t:{score}\n")

print(f"CV_SCORES_mean: {np.mean(scores)}")
print(f"CV_SCORES_std: {np.std(scores)}")

Fold 0	:329.57939730174485

Fold 1	:121.5917109821461

Fold 2	:100.86816295980721

Fold 3	:111.54476276767218

Fold 4	:130.18600581877877

Fold 5	:413.1955657005038

Fold 6	:753.5164905050091

Fold 7	:131.0299002140328

Fold 8	:154.53780950316332

Fold 9	:391.98973108727034

CV_SCORES_mean: 263.8039536840128
CV_SCORES_std: 200.06780740402974


In [196]:
scores=[]
for i,j in enumerate(cv.split(X_a_encode,y_a)):
    model=LGBMRegressor(boosting_type='gbdt', objective='regression',seed=RANDOM_SEED)
    train_X=X_a_encode.iloc[j[0]] 
    train_y=np.log1p(y_a.iloc[j[0]])
    test_X=X_a_encode.iloc[j[1]]
    test_y=np.log1p(y_a.iloc[j[1]])
    model.fit(train_X,train_y,
        eval_names=['train', 'valid'],
        eval_set=[(train_X,train_y),(test_X, test_y)], 
        eval_metric=rmse_lgb,
        verbose = False)
    
    y_pred=model.predict(X_a_encode.iloc[j[1]])
    score=rmse(np.expm1(y_pred),y_a.iloc[j[1]])
    scores.append(score)
    print(f"Fold {i}\t:{score}\n")

print(f"CV_SCORES_mean: {np.mean(scores)}")
print(f"CV_SCORES_std: {np.std(scores)}")

Fold 0	:629.2172785194281

Fold 1	:1084.3066797251786

Fold 2	:458.0473260438019

Fold 3	:1045.0848005362018

Fold 4	:684.8382768978781

Fold 5	:465.02118982546125

Fold 6	:450.7220825204008

Fold 7	:226.26301819555005

Fold 8	:601.5455582865802

Fold 9	:1825.196716513562

CV_SCORES_mean: 747.0242927064044
CV_SCORES_std: 438.86052998515385


In [197]:
model_1 = LGBMRegressor(boosting_type='gbdt', objective='regression',seed=RANDOM_SEED)
model_1.fit(X_n_encode, y_n)

model_2 = LGBMRegressor(boosting_type='gbdt', objective='regression',seed=RANDOM_SEED)
model_2.fit(X_a_encode, y_a)

LGBMRegressor(objective='regression', seed=0)

## Submission

In [211]:
sub = pd.read_csv("data/SampleSubmission.csv")
test = pd.read_csv("data/Test.csv")
test.drop(["ID"], axis=1, inplace=True)

In [212]:
test = pre_processing(test)

test = catboost_encoder_df.transform(test)


In [213]:
labels_1 = lof.predict(test)

test_anomalies = test[labels_1 == -1]

test_anomalies.head()

,District,Block,CultLand,CropCultLand,LandPreparationMethod,CropTillageDate,CropTillageDepth,CropEstMethod,RcNursEstDate,SeedingSowingTransplanting,SeedlingsPerPit,NursDetFactor,TransDetFactor,TransplantingIrrigationHours,TransplantingIrrigationSource,TransplantingIrrigationPowerSource,TransIrriCost,StandingWater,OrgFertilizers,Ganaura,CropOrgFYM,PCropSolidOrgFertAppMethod,NoFertilizerAppln,CropbasalFerts,BasalDAP,BasalUrea,MineralFertAppMethod,FirstTopDressFert,1tdUrea,1appDaysUrea,2tdUrea,2appDaysUrea,MineralFertAppMethod.1,Harv_method,Harv_date,Harv_hand_rent,Threshing_date,Threshing_method,Residue_length,Residue_perc,Stubble_use,Acre,1tdUrea_M,1appDaysUrea_2,2tdUrea_M,2appDaysUrea_M,CropTillageMonth,RcNursEstMonth,SeedingSowingTransplantingMonth,Harv_Month,Harv_date-Threshing_date,RcNursEstDate-SeedingSowingTransplanting,CropTillageDate-SeedingSowingTransplanting,CropTillageDate-RcNursEstDate,RcNursEstDate-Threshing_date,SeedingSowingTransplanting-Threshing_date,Harv_date-SeedingSowingTransplanting,CropTillageDate-Threshing_date,CropTillageDate-Harv_date,num_LandPreparationMethod,num_FirstTopDressFert,num_CropbasalFerts,num_TransDetFactor,num_NursDetFactor
0,627.197281,770.324117,20,13,620.630983,665.967145,4,1040.396116,669.226091,683.523827,1.0,539.287360,579.762453,2.0,543.842173,465.612724,200.0,2.0,534.152418,0.0,2.0,554.174135,3,559.708505,6.0,0.0,540.016725,526.475255,8.0,23.0,6.0,63.0,542.002718,532.086199,839.925747,500.0,563.254789,524.887982,22,10,537.902924,0.272727,0,0,0,0,589.329769,542.993703,555.866469,918.021108,26,26,5,-21,178,152,-126,157,131,1,1,1,4,3
1,627.197281,770.324117,25,25,620.630983,665.967145,5,519.718194,390.967065,555.498271,4.0,443.174362,140.160920,2.0,445.553356,555.434133,0.0,3.0,541.457354,0.0,0.0,554.174135,3,559.708505,5.0,0.0,540.016725,526.475255,7.0,23.0,8.0,65.0,542.002718,532.086199,375.355235,150.0,734.101680,550.945189,28,10,537.902924,0.227273,0,0,0,0,589.329769,542.993703,496.617471,502.115448,38,25,9,-16,165,140,-102,149,111,1,1,1,4,3
8,253.971272,558.285824,15,15,433.153350,486.153350,6,519.718194,561.099827,217.820191,2.0,659.935215,345.066929,15.0,543.842173,465.612724,1550.0,2.0,541.457354,0.0,0.0,554.174135,2,381.547893,0.0,35.0,533.573264,539.287360,38.0,15.0,-999.0,-999.0,542.002718,532.086199,236.857280,2000.0,344.657472,524.887982,27,10,537.902924,0.681818,0,0,1,1,589.329769,537.370348,555.866469,384.618370,10,30,7,-23,128,98,-88,105,95,1,2,2,1,1
10,658.009195,684.785720,12,12,473.536579,514.297498,4,519.718194,529.207273,628.877952,2.0,581.493704,802.875434,2.0,543.842173,555.434133,100.0,2.0,541.457354,0.0,0.0,554.174135,2,566.631468,0.0,5.0,540.016725,652.785720,7.0,30.0,-999.0,-999.0,542.002718,532.086199,368.882059,250.0,1383.485824,550.945189,23,10,537.902924,0.156250,0,0,1,1,589.329769,537.370348,555.866469,384.618370,65,20,4,-16,196,176,-111,180,115,2,2,1,3,3
14,627.197281,497.731044,10,10,619.085926,366.382613,2,519.718194,333.881344,555.498271,4.0,620.951053,611.337118,9.0,543.842173,555.434133,3000.0,1.0,541.457354,4.0,0.0,481.454147,2,528.963206,10.0,10.0,540.016725,681.857472,0.0,0.0,-999.0,-999.0,542.002718,532.086199,573.340175,500.0,540.134551,550.945189,26,10,537.902924,0.454545,0,0,1,1,367.862624,542.993703,496.617471,502.115448,56,23,38,15,192,169,-113,207,151,3,1,2,1,1


In [225]:
pred_1 = model_1.predict(test[labels_1 == 1])
pred_2 = model_1.predict(test[labels_1 == -1])

In [228]:
predict = model.predict(test)
sub.at[labels_1==1, "Yield"]=pred_1
sub.at[labels_1==-1, "Yield"]=pred_2/10
sub.head()

,ID,Yield
0,ID_F9XXEXN2ADR2,69.788844
1,ID_SO3VW2X4QO93,26.036536
2,ID_UKUQ7JM8E894,425.441295
3,ID_QUISMWEZR2H4,294.940041
4,ID_25JGI455VKCZ,517.530959


In [229]:
sub.to_csv('submission_10.csv', index = False)